# Sistema de Apoyo para Medicos Rurales en Colombia
## Setup del Modelo Base: `google/mt5-small` con LoRA (PEFT)

**Curso:** Topicos Especiales y Aplicaciones en IA  
**Objetivo:** Fine-tunear mT5-small para generar protocolos de tratamiento a partir de descripciones de sintomas, basado en las guias clinicas del Ministerio de Salud de Colombia.

### Por que `google/mt5-small`?
- **Multilingue:** Pre-entrenado en 101 idiomas, incluyendo espanol.
- **Tamano manejable:** ~300M parametros cabe en la T4 gratuita de Colab (~15GB VRAM).
- **Arquitectura Seq2Seq:** Ideal para tareas de generacion de texto (sintomas a protocolo).
- **Licencia:** Apache 2.0 (uso comercial y academico permitido).
- **Evidencia:** Fine-tuning exitoso documentado en espanol y dominios medicos.

### Estrategia de entrenamiento: LoRA (Low-Rank Adaptation)
Con LoRA solo entrenamos una fraccion pequena de los parametros del modelo, lo que reduce la memoria necesaria y el tiempo de entrenamiento sin sacrificar calidad.

---
> **Nota:** Este notebook es autocontenido. Ejecutalo de arriba a abajo en una sola sesion de Colab con runtime de GPU (T4).

## Paso 1: Instalacion de Dependencias

Instalamos todas las librerias necesarias:
- `transformers`: Para cargar mT5 y el tokenizador.
- `datasets`: Para manejar el dataset de entrenamiento.
- `peft`: Para aplicar LoRA al modelo.
- `accelerate`: Para optimizar el entrenamiento en GPU.
- `bitsandbytes`: Para cuantizacion (util para optimizar memoria).
- `sentencepiece`: **Requerido** por el tokenizador de mT5 (usa SentencePiece internamente).
- `torch`: Framework de deep learning (ya incluido en Colab, pero lo declaramos explicitamente).

In [ ]:
# Instalacion silenciosa de todas las dependencias del proyecto
# El flag -q suprime la salida verbosa de pip para mayor legibilidad
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece torch

print('Dependencias instaladas correctamente.')

## Paso 2: Importaciones Explicitas

Importamos cada modulo de forma explicita para mayor claridad y para evitar problemas de namespace.

In [ ]:
# --- Librerias estandar de Python ---
import os
import json

# --- PyTorch: framework de deep learning ---
import torch

# --- Hugging Face Transformers: modelo y tokenizador ---
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Hugging Face Datasets: carga y manejo de datasets ---
from datasets import load_dataset, Dataset

# --- PEFT: Parameter-Efficient Fine-Tuning (LoRA) ---
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Verificar versiones instaladas
import transformers
import peft
import datasets as datasets_lib

print('Importaciones completadas.')
print(f'   PyTorch version      : {torch.__version__}')
print(f'   Transformers version : {transformers.__version__}')
print(f'   PEFT version         : {peft.__version__}')
print(f'   Datasets version     : {datasets_lib.__version__}')

## Paso 3: Configuracion del Dispositivo (CPU / GPU)

Detectamos si hay GPU disponible. En Colab con runtime T4, `torch.cuda.is_available()` debe retornar `True`.

> **Importante:** Si ves `Dispositivo: cpu`, ve a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU (T4)`.

In [ ]:
# Detectar dispositivo disponible: GPU (CUDA) o CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Dispositivo seleccionado: {device}')

if torch.cuda.is_available():
    # Mostrar informacion detallada de la GPU
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # Convertir a GB
    print(f'   GPU            : {gpu_name}')
    print(f'   VRAM disponible: {vram_total:.2f} GB')
else:
    print('   ADVERTENCIA: No se detecto GPU. El entrenamiento sera muy lento en CPU.')
    print('   Considera activar el runtime de GPU en Colab.')

## Paso 4: Carga del Modelo y Tokenizador

Cargamos `google/mt5-small` desde Hugging Face Hub.

- **`AutoTokenizer`**: Carga el tokenizador SentencePiece de mT5.
- **`AutoModelForSeq2SeqLM`**: Carga la arquitectura encoder-decoder de mT5 con cabeza de generacion de lenguaje.

La primera vez tardara 1-2 minutos mientras descarga los pesos (~1.2 GB).

In [ ]:
# Identificador del modelo en Hugging Face Hub
MODEL_NAME = 'google/mt5-small'

print(f'Cargando tokenizador desde "{MODEL_NAME}"...')

# Cargar el tokenizador de mT5
# mT5 usa SentencePiece, por eso instalamos el paquete 'sentencepiece' en el Paso 1
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Tokenizador cargado.')
print(f'\nInformacion del Tokenizador:')
print(f'   Tipo                 : {type(tokenizer).__name__}')
print(f'   Tamano del vocabulario: {tokenizer.vocab_size:,} tokens')
print(f'   Token de padding     : "{tokenizer.pad_token}" (ID: {tokenizer.pad_token_id})')
print(f'   Token EOS            : "{tokenizer.eos_token}" (ID: {tokenizer.eos_token_id})')
print(f'   Usa SentencePiece    : {hasattr(tokenizer, "sp_model")}')

In [ ]:
print(f'Cargando modelo "{MODEL_NAME}"... (puede tomar 1-2 minutos)')

# Cargar el modelo mT5-small con cabeza de generacion Seq2Seq
# torch_dtype=torch.float32 para compatibilidad maxima con la T4 gratuita
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32  # float16 tambien es valido para ahorrar VRAM
)

# Mover el modelo al dispositivo (GPU si esta disponible, de lo contrario CPU)
model = model.to(device)

print(f'Modelo cargado y movido a {device}.')

# Calcular el numero total de parametros del modelo base
total_params = sum(p.numel() for p in model.parameters())
print(f'\nEstadisticas del Modelo Base:')
print(f'   Parametros totales: {total_params:,} ({total_params/1e6:.1f}M)')
print(f'   Arquitectura      : {type(model).__name__}')

## Paso 5: Verificacion del Tokenizador

Probamos el tokenizador con una frase de ejemplo en espanol para confirmar que funciona correctamente y que el vocabulario multilingue esta activo.

In [ ]:
# Frase de prueba representativa del dominio del proyecto
frase_ejemplo = 'Paciente con fiebre y tos'

print(f'Tokenizando frase de ejemplo: "{frase_ejemplo}"\n')

# Tokenizar la frase (return_tensors='pt' retorna tensores de PyTorch)
tokens_encoding = tokenizer(
    frase_ejemplo,
    return_tensors='pt',
    padding=True
)

# Extraer los IDs de tokens
token_ids = tokens_encoding['input_ids'][0].tolist()

# Convertir cada ID de vuelta a su token de texto para visualizacion
tokens_texto = tokenizer.convert_ids_to_tokens(token_ids)

print('Resultado de la tokenizacion:')
print(f'   Frase original : {frase_ejemplo}')
print(f'   Tokens         : {tokens_texto}')
print(f'   IDs de tokens  : {token_ids}')
print(f'   Cantidad tokens: {len(token_ids)}')

# Decodificar de vuelta para verificar la reconstruccion
frase_reconstruida = tokenizer.decode(token_ids, skip_special_tokens=True)
print(f'\nDecodificacion de vuelta: "{frase_reconstruida}"')
print(f'   Reconstruccion correcta: {frase_reconstruida.strip() == frase_ejemplo}')

## Paso 6: Inferencia de Prueba con el Modelo Base (Sin Fine-Tuning)

Ejecutamos una inferencia de prueba para confirmar que el modelo funciona correctamente en el pipeline completo.

> **Nota:** La salida generada en este punto **no sera util medicamente** - el modelo base no ha sido entrenado en protocolos clinicos colombianos. El objetivo es unicamente verificar que el pipeline funciona de extremo a extremo.

In [ ]:
# Texto de entrada de prueba: descripcion de sintomas
# En el sistema final, este seria el input del medico rural
texto_entrada = 'Sintomas: Paciente adulto con fiebre de 39C, tos seca persistente y dificultad respiratoria leve. Protocolo de tratamiento:'

print('Realizando inferencia de prueba con el modelo base...')
print(f'   Entrada: {texto_entrada}\n')

# Tokenizar el texto de entrada y moverlo al mismo dispositivo que el modelo
inputs = tokenizer(
    texto_entrada,
    return_tensors='pt',
    max_length=128,
    truncation=True
).to(device)

# Generacion de texto con model.generate()
# Usamos torch.no_grad() para deshabilitar el calculo de gradientes (no estamos entrenando)
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=64,        # Maximo de tokens a generar
        num_beams=4,              # Busqueda en haz (beam search) para mayor calidad
        early_stopping=True,      # Parar cuando todos los beams llegan al EOS
        no_repeat_ngram_size=2,   # Evitar repeticion de bigramas
    )

# Decodificar la salida generada
texto_generado = tokenizer.decode(outputs[0], skip_special_tokens=True)

print('Salida del modelo base (sin fine-tuning):')
print(f'   "{texto_generado}"')
print('\n   NOTA: Salida esperada: incoherente o en otro idioma.')
print('   Esto es normal. El modelo necesita fine-tuning para generar protocolos medicos.')
print('\nPipeline de inferencia funcionando correctamente.')

## Paso 7: Configuracion de LoRA con PEFT

Aplicamos **LoRA (Low-Rank Adaptation)** para hacer el fine-tuning eficiente en parametros.

### Por que LoRA?
En lugar de actualizar todos los ~300M parametros del modelo, LoRA inyecta matrices de bajo rango en las capas de atencion. Solo estas matrices pequenas se entrenan, reduciendo el costo computacional drasticamente.

### Hiperparametros seleccionados:
| Parametro | Valor | Razon |
|-----------|-------|-------|
| `r` | 8 | Rango de las matrices LoRA. Mayor rango = mas capacidad, mas parametros. |
| `lora_alpha` | 16 | Factor de escala. Generalmente alpha = 2*r. |
| `lora_dropout` | 0.05 | Regularizacion leve para evitar overfitting. |
| `target_modules` | `["q", "v"]` | Aplicar LoRA solo a las proyecciones Query y Value de la atencion. |

In [ ]:
# Definir la configuracion de LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,   # Tipo de tarea: generacion Seq2Seq (Encoder-Decoder)
    r=8,                                # Rango de las matrices de bajo rango
    lora_alpha=16,                      # Factor de escala de LoRA (alpha = 2*r es buen punto de partida)
    lora_dropout=0.05,                  # Dropout aplicado a las capas LoRA (regularizacion)
    target_modules=['q', 'v'],          # Modulos de atencion a adaptar: Query y Value
    bias='none',                        # No adaptar los sesgos (reduce parametros adicionales)
)

print('Configuracion de LoRA:')
print(f'   Tarea          : {lora_config.task_type}')
print(f'   Rango (r)      : {lora_config.r}')
print(f'   Alpha          : {lora_config.lora_alpha}')
print(f'   Dropout        : {lora_config.lora_dropout}')
print(f'   Modulos target : {lora_config.target_modules}')

# Aplicar la configuracion LoRA al modelo base
# get_peft_model() envuelve el modelo base e inyecta los adaptadores LoRA
model = get_peft_model(model, lora_config)

print('\nAdaptadores LoRA inyectados en el modelo.')

In [ ]:
# Calcular y mostrar la cantidad de parametros entrenables vs totales
# Esto demuestra la eficiencia de LoRA frente al fine-tuning completo

params_totales = sum(p.numel() for p in model.parameters())
params_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_congelados = params_totales - params_entrenables
porcentaje_entrenables = 100 * params_entrenables / params_totales

print('Resumen de Parametros del Modelo con LoRA:')
print(f'   Parametros totales     : {params_totales:,}')
print(f'   Parametros congelados  : {params_congelados:,}')
print(f'   Parametros entrenables : {params_entrenables:,}')
print(f'   Porcentaje entrenable  : {porcentaje_entrenables:.4f}%')
print(f'\nCon LoRA solo entrenamos el {porcentaje_entrenables:.4f}% de los parametros.')
print('   Reduccion de VRAM y tiempo de entrenamiento significativa.')

# Vista estructural usando el metodo de PEFT (informativo)
model.print_trainable_parameters()

## Paso 8: Preparacion del Dataset

### Estructura esperada del dataset

El dataset debe estar en formato **CSV** o **JSON Lines** con dos columnas:

| Columna | Descripcion | Ejemplo |
|---------|-------------|--------|
| `input` | Descripcion de sintomas del paciente | `"Paciente con fiebre de 39C, tos seca y dificultad respiratoria leve"` |
| `output` | Protocolo de tratamiento o estudio a solicitar | `"Solicitar: hemograma, PCR. Tratar con acetaminofen 500mg c/8h. Referir si SpO2 < 94%."` |

Los datos deben provenir de las **Guias de Practica Clinica del Ministerio de Salud de Colombia**.

---

> **Nota:** La celda de abajo contiene el codigo de carga **comentado**. Descomenta el bloque correspondiente al formato de tu archivo cuando el dataset este listo.

In [ ]:
# ============================================================
#  CELDA DE CARGA DE DATASET (descomentar cuando este listo)
# ============================================================

# --- OPCION A: Cargar desde un archivo CSV ---
# El CSV debe tener cabecera con columnas 'input' y 'output'
#
# Ejemplo de estructura del CSV:
#   input,output
#   "Paciente con fiebre de 39C y tos seca","Solicitar hemograma. Administrar acetaminofen 500mg c/8h."
#
# dataset = load_dataset(
#     'csv',
#     data_files={'train': 'dataset_medico_rural.csv'},
#     split='train'
# )


# --- OPCION B: Cargar desde un archivo JSON Lines ---
# Cada linea del archivo es un JSON con campos 'input' y 'output'
#
# Ejemplo de estructura del JSONL:
#   {"input": "Paciente con fiebre", "output": "Solicitar hemograma."}
#
# dataset = load_dataset(
#     'json',
#     data_files={'train': 'dataset_medico_rural.jsonl'},
#     split='train'
# )


# --- OPCION C: Cargar desde Google Drive (recomendado en Colab) ---
# from google.colab import drive
# drive.mount('/content/drive')
# dataset = load_dataset(
#     'csv',
#     data_files={'train': '/content/drive/MyDrive/dataset_medico_rural.csv'},
#     split='train'
# )


# --- Dataset de ejemplo minimo en memoria (para probar el pipeline) ---
# Util para verificar la tokenizacion antes de tener el dataset real
datos_ejemplo = [
    {
        'input': 'Paciente masculino de 45 anos con fiebre de 39C, tos seca y dificultad respiratoria leve.',
        'output': 'Solicitar: hemograma completo, PCR cuantitativa, radiografia de torax. '
                  'Tratar sintomas con acetaminofen 500mg c/8h. Monitorizar SpO2. '
                  'Referir a urgencias si SpO2 < 94% o disnea progresiva.'
    },
    {
        'input': 'Paciente femenina de 32 anos con dolor abdominal tipo colico en cuadrante inferior derecho, '
                 'nauseas y fiebre de 38.2C.',
        'output': 'Solicitar: hemograma, proteina C reactiva, ecografia abdominal urgente. '
                  'Evaluar signos de Blumberg y McBurney. '
                  'Si ecografia confirma apendicitis: referencia urgente a cirugia.'
    },
    {
        'input': 'Nino de 7 anos con exantema macular eritematoso en tronco, fiebre de 38.5C y coriza.',
        'output': 'Sospecha de sarampion. Solicitar: IgM antisarampion. '
                  'Aislamiento respiratorio inmediato. Notificacion obligatoria al SIVIGILA. '
                  'Soporte sindromatico: hidratacion, acetaminofen. Referir si complicaciones.'
    },
]

# Crear dataset en memoria a partir de los datos de ejemplo
dataset_ejemplo = Dataset.from_list(datos_ejemplo)

print('Dataset de ejemplo cargado en memoria:')
print(f'   Numero de ejemplos : {len(dataset_ejemplo)}')
print(f'   Columnas           : {dataset_ejemplo.column_names}')
print(f'\nPrimer ejemplo:')
print(f'   INPUT  : {dataset_ejemplo[0]["input"]}')
print(f'   OUTPUT : {dataset_ejemplo[0]["output"]}')

## Paso 9: Funcion de Preprocesamiento del Dataset

Definimos la funcion que tokeniza los pares `(sintomas, protocolo)` para alimentar al modelo durante el entrenamiento.

Para un modelo Seq2Seq:
- **`input_ids`**: Tokens del texto de entrada (sintomas) procesados por el **encoder**.
- **`labels`**: Tokens del texto de salida (protocolo) generados por el **decoder**.

In [ ]:
# Longitudes maximas de secuencia
# Ajustar segun la longitud real de los protocolos del dataset
MAX_INPUT_LENGTH = 256    # Maximo de tokens para la descripcion de sintomas
MAX_TARGET_LENGTH = 512   # Maximo de tokens para el protocolo de tratamiento

def preprocesar_ejemplo(ejemplos):
    """
    Tokeniza los pares (sintomas, protocolo) para el entrenamiento Seq2Seq.

    Args:
        ejemplos: Batch de ejemplos con campos 'input' y 'output'.

    Returns:
        Diccionario con input_ids, attention_mask y labels tokenizados.
    """
    # Tokenizar el texto de entrada (sintomas del paciente)
    model_inputs = tokenizer(
        ejemplos['input'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length'  # Padding al maximo para batches uniformes
    )

    # Tokenizar el texto de salida (protocolo de tratamiento) como labels
    labels = tokenizer(
        text_target=ejemplos['output'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding='max_length'
    )

    # Los tokens de padding en los labels se convierten a -100
    # para que la funcion de perdida los ignore durante el calculo
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]

    model_inputs['labels'] = labels_ids
    return model_inputs


# Aplicar el preprocesamiento al dataset de ejemplo
dataset_tokenizado = dataset_ejemplo.map(
    preprocesar_ejemplo,
    batched=True,                              # Procesar en lotes para mayor eficiencia
    remove_columns=['input', 'output']         # Eliminar columnas de texto original
)

print('Dataset tokenizado correctamente.')
print(f'   Columnas disponibles       : {dataset_tokenizado.column_names}')
print(f'   Longitud de input_ids[0]   : {len(dataset_tokenizado[0]["input_ids"])} tokens')
print(f'   Longitud de labels[0]      : {len(dataset_tokenizado[0]["labels"])} tokens')

## Paso 10: Guardado y Carga del Adaptador LoRA

Una vez completado el fine-tuning, guardamos **solo los adaptadores LoRA** (no todo el modelo), lo que resulta en archivos mucho mas pequenos (~10-50 MB vs ~1.2 GB del modelo completo).

In [ ]:
# Directorio donde se guardaran los adaptadores LoRA entrenados
DIRECTORIO_ADAPTADOR = './mt5_medico_rural_lora'

# --- GUARDAR el adaptador LoRA despues del entrenamiento ---
# model.save_pretrained() guarda SOLO los pesos de LoRA,
# no los pesos del modelo base (estos se descargan de HuggingFace al cargar)
model.save_pretrained(DIRECTORIO_ADAPTADOR)
tokenizer.save_pretrained(DIRECTORIO_ADAPTADOR)  # Tambien guardar el tokenizador

print(f'Adaptador LoRA guardado en: "{DIRECTORIO_ADAPTADOR}"')

# Listar los archivos guardados con su tamano
archivos_guardados = os.listdir(DIRECTORIO_ADAPTADOR)
print(f'\nArchivos en el directorio del adaptador:')
for archivo in archivos_guardados:
    ruta_completa = os.path.join(DIRECTORIO_ADAPTADOR, archivo)
    tamano_kb = os.path.getsize(ruta_completa) / 1024
    print(f'   {archivo:<45} ({tamano_kb:.1f} KB)')

In [ ]:
# --- CARGAR el adaptador LoRA en una sesion futura ---
# Este bloque muestra como reutilizar el modelo fine-tuneado
# sin necesidad de repetir el entrenamiento

print('Cargando modelo con adaptador LoRA entrenado...')

# Paso 1: Cargar el modelo base desde Hugging Face Hub
modelo_base_recargado = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

# Paso 2: Cargar el tokenizador guardado junto con el adaptador
tokenizer_recargado = AutoTokenizer.from_pretrained(DIRECTORIO_ADAPTADOR)

# Paso 3: Envolver el modelo base con el adaptador LoRA guardado
# PeftModel.from_pretrained() inyecta los pesos de LoRA en el modelo base
modelo_con_lora = PeftModel.from_pretrained(
    modelo_base_recargado,
    DIRECTORIO_ADAPTADOR
)

# Mover al dispositivo y poner en modo evaluacion (desactiva dropout)
modelo_con_lora = modelo_con_lora.to(device)
modelo_con_lora.eval()

print(f'Modelo con adaptador LoRA recargado correctamente en {device}.')
print(f'   Tipo del modelo: {type(modelo_con_lora).__name__}')
print('\nEl modelo esta listo para hacer inferencias con el adaptador entrenado.')

## Paso 11: Verificacion Final del Setup

Resumen completo del estado del sistema antes de iniciar el entrenamiento.

In [ ]:
# Verificacion final de todos los componentes del sistema

print('=' * 62)
print('  SISTEMA DE APOYO PARA MEDICOS RURALES EN COLOMBIA')
print('  Setup Completo - Verificacion Final')
print('=' * 62)

# 1. Verificar GPU
gpu_ok = torch.cuda.is_available()
estado_gpu = 'OK' if gpu_ok else 'ADVERTENCIA'
print(f'\n [{estado_gpu}] GPU disponible: {gpu_ok}')
if gpu_ok:
    vram_libre = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated()) / (1024**3)
    print(f'         GPU    : {torch.cuda.get_device_name(0)}')
    print(f'         VRAM libre: {vram_libre:.2f} GB')

# 2. Verificar modelo
print(f'\n [OK] Modelo cargado: {MODEL_NAME}')
print(f'         Tipo: {type(model).__name__}')

# 3. Verificar tokenizador
print(f'\n [OK] Tokenizador listo')
print(f'         Vocabulario: {tokenizer.vocab_size:,} tokens')
print(f'         SentencePiece: {hasattr(tokenizer, "sp_model")}')

# 4. Verificar LoRA
params_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
params_total = sum(p.numel() for p in model.parameters())
print(f'\n [OK] LoRA configurado (r=8, alpha=16, dropout=0.05)')
print(f'         Target modules: [q, v]')
print(f'         Parametros entrenables: {params_train:,} ({100*params_train/params_total:.4f}%)')

# 5. Verificar dataset
print(f'\n [OK] Estructura de dataset verificada')
print(f'         Dataset de ejemplo: {len(dataset_ejemplo)} muestras')
print(f'         Dataset tokenizado: listo para Trainer')

# 6. Verificar guardado
lora_saved = os.path.exists(os.path.join(DIRECTORIO_ADAPTADOR, 'adapter_config.json'))
print(f'\n [OK] Sistema de guardado/carga verificado')
print(f'         Adaptador LoRA guardado: {lora_saved}')
print(f'         Directorio: {DIRECTORIO_ADAPTADOR}')

print('\n' + '=' * 62)
print('  SETUP COMPLETO')
print('  El modelo esta listo para fine-tuning.')
print('  Proximo paso: cargar el dataset real y configurar')
print('  el Trainer de Hugging Face para el entrenamiento.')
print('=' * 62)

---

## Proximos Pasos

1. **Construir el dataset real** a partir de las Guias Clinicas del Ministerio de Salud de Colombia.
2. **Reemplazar el dataset de ejemplo** en el Paso 8 con el dataset real usando `load_dataset()`.
3. **Configurar el `Trainer`** de Hugging Face con hiperparametros de entrenamiento (learning rate, epochs, batch size).
4. **Ejecutar el fine-tuning** y monitorear la perdida de entrenamiento.
5. **Evaluar** el modelo fine-tuneado con metricas apropiadas (ROUGE, BERTScore, o evaluacion humana de protocolos).
6. **Desplegar** el adaptador LoRA junto con el modelo base para inferencia en produccion.

---
*Topicos Especiales y Aplicaciones en IA - Sistema de Apoyo para Medicos Rurales en Colombia*